# **Library package**

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
library(purrr)

In [ ]:
library()

# **Import data**

In [3]:
val_dat<-readRDS("val_result.rds")

# **Run the codes**

In [5]:
vars <- c("rifam", "pyra", "iso", "etham", "med", "place",
          "swollow", "protude", "seq", "move", "complete")

# 2. Manual calculation function in percentages using Wilson Method
calc_diag_percent <- function(var_name, data) {

  test_val <- data[[paste0("t_", var_name)]]
  gold_val <- data[[paste0("g_", var_name)]]

  # Calculate 2x2 components
  tp <- sum(test_val == TRUE & gold_val == TRUE, na.rm = TRUE)
  fp <- sum(test_val == TRUE & gold_val == FALSE, na.rm = TRUE)
  fn <- sum(test_val == FALSE & gold_val == TRUE, na.rm = TRUE)
  tn <- sum(test_val == FALSE & gold_val == FALSE, na.rm = TRUE)

  # Helper for proportions to Percent with 95% CI (Wilson Method)
  get_ci_percent <- function(x, n) {
    if (n == 0) return("N/A")

    # prop.test performs the Wilson score interval
    # correct = FALSE provides the standard Wilson method
    res <- prop.test(x, n, conf.level = 0.95, correct = FALSE)

    # Extract values
    est <- round(res$estimate * 100, 1)
    low <- round(res$conf.int[1] * 100, 1)
    high <- round(res$conf.int[2] * 100, 1)

    return(sprintf("%.1f%% (%.1f%%, %.1f%%)", est, low, high))
  }

  # Create a row for this variable
  data.frame(
    Variable = var_name,
    Sensitivity = get_ci_percent(tp, tp + fn),
    Specificity = get_ci_percent(tn, tn + fp),
    PPV = get_ci_percent(tp, tp + fp),
    NPV = get_ci_percent(tn, tn + fn),
    Accuracy = get_ci_percent(tp + tn, tp + tn + fp + fn),
    stringsAsFactors = FALSE
  )
}

# 3. Apply to all variables
final_table_percent <- map_df(vars, ~calc_diag_percent(.x, val_dat))

Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”
Warning message in prop.test(x, n, conf.level = 0.95, correct = FALSE):
“Chi-squared approximation may be incorrect”


# **Show the results**

In [6]:
print(final_table_percent)

   Variable            Sensitivity            Specificity
1     rifam   79.7% (72.4%, 85.5%)   50.0% (29.0%, 71.0%)
2      pyra   87.6% (80.0%, 92.6%)   94.6% (85.4%, 98.2%)
3       iso   88.2% (82.2%, 92.4%)   87.5% (52.9%, 97.8%)
4     etham   82.1% (73.7%, 88.2%)   58.2% (45.0%, 70.3%)
5       med   72.6% (62.9%, 80.6%)   86.4% (76.1%, 92.7%)
6     place   68.6% (61.0%, 75.3%) 100.0% (51.0%, 100.0%)
7   swollow   78.0% (70.9%, 83.7%) 100.0% (43.9%, 100.0%)
8   protude 100.0% (75.8%, 100.0%) 100.0% (97.5%, 100.0%)
9       seq   90.6% (84.7%, 94.5%)   90.9% (72.2%, 97.5%)
10     move 100.0% (56.6%, 100.0%)   99.4% (96.5%, 99.9%)
11 complete   60.0% (23.1%, 88.2%) 100.0% (97.6%, 100.0%)
                      PPV                    NPV               Accuracy
1    92.7% (86.7%, 96.1%)   23.7% (13.0%, 39.2%)   76.4% (69.3%, 82.3%)
2    96.8% (91.1%, 98.9%)   80.3% (69.2%, 88.1%)   90.1% (84.5%, 93.8%)
3    99.3% (96.0%, 99.9%)   28.0% (14.3%, 47.6%)   88.2% (82.3%, 92.3%)
4    79.1% (70.6